### This notebook takes previously extracted environmental sentences from Annual Information Forms (AIFs) and labels each sentence along three dimensions: **repetitive vs. novel**, **specific vs. general**, and **useful vs. fluffy**. The notebook assumes that the environmental sentences have already been extracted using the keywords "environment, environmental, restoration, reclamation, decommissioning" and does not include the sentence extraction step.

In [ ]:
# @title We start by importing the essential libraries
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.util import ngramsw
from collections import Counter
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')



In [ ]:
# @title FUNCTIONS

# Remove short capitalized words (fewer than 4 characters) from a text string.
# This is primarily used to eliminate company tickers and abbreviated company
# names (e.g., "ABC", "XYZ") that may interfere with semantic comparisons
# between sentences.

def remove_123_words(string):
    # Split the text into individual words
    words = string.split()

    # Keep only words that are not both:
    # (1) capitalized at the first character, and
    # (2) shorter than 4 characters
    result = [word for word in words if not (word[0].isupper() and len(word) < 4)]

    # Reconstruct the cleaned text
    return ' '.join(result)


# Remove extra whitespace from a text string.
# Consecutive spaces, tabs, and line breaks are replaced with a single space,
# and leading/trailing whitespace is removed.
def remove_whitespace(text):
    return " ".join(text.split())



# Create a column containing the environmental disclosure from the
# previous reporting period for the same company and disclosure group.
# This allows each disclosure to be compared with its prior-year version
# when assessing whether the content is repeated or novel.

def manipulate_data(df):
    # Group observations by company and disclosure group
    grouped = df.groupby(['company', 'group'])
    # For each company-group combination, shift the environmental
    # disclosure by one period to obtain the previous year's disclosure
    df['environment_previous'] = grouped['environment'].shift(+1)

    return df


In [ ]:
# Create two binary indicators that classify firms as large or small
# based on their market capitalization. One indicator uses the sample
# median market capitalization as the threshold, while the other uses
# the sample mean.

# Indicator based on the median market capitalization
for index, row in df.iterrows():
    if row['Market.Cap'] > df['Market.Cap'].median():
        df.at[index, 'big.mktcap.median'] = 1
    else:
        df.at[index, 'big.mktcap.median'] = 0

# Indicator based on the mean market capitalization
for index, row in df.iterrows():
    if row['Market.Cap'] > df['Market.Cap'].mean():
        df.at[index, 'big.mktcap.mean'] = 1
    else:
        df.at[index, 'big.mktcap.mean'] = 0

## This preprocessing step prepares the dataset for longitudinal disclosure analysis. Observations are organized by company and reporting year, and consecutive reporting sequences are identified. Firm-years that cannot be linked to a prior-year disclosure are removed. For each remaining observation, the environmental disclosure from the previous reporting year is added to a new column (environment_previous), creating a paired structure that enables comparison of current and prior disclosures for repeated-versus-novel analysis.

In [ ]:
# Prepare the dataset for longitudinal comparison of environmental disclosures.
# The code:
# (1) retains relevant variables,
# (2) converts filing dates to reporting years,
# (3) identifies consecutive reporting sequences for each company,
# (4) removes isolated firm-years that lack a prior observation,
# (5) creates a column containing the previous year's disclosure, and
# (6) retains only observations that can be compared with a prior-year disclosure.

# Keep only variables required for the analysis
df = df.loc[:, ['Accession.Number', 'Issuer.Name', 'Filing.Date',
                'Market.Cap', 'big.mktcap.median',
                'big.mktcap.mean', 'environment']]

# Convert filing dates to datetime format and extract the year
df['Filing.Date'] = pd.to_datetime(df['Filing.Date'])

# Rename variables for consistency
df = df.rename(columns={"Issuer.Name": 'company',
                        'Filing.Date': 'year'})

# Retain only the reporting year
df['year'] = df['year'].dt.year

# Sort observations chronologically within each company
df.sort_values(['company', 'year'], inplace=True)
df.reset_index(drop=True, inplace=True)

# Calculate the gap between consecutive reporting years
df.sort_values(['company', 'year'], inplace=True)
df['year_diff'] = df.groupby('company')['year'].diff()

# Identify the start of a new sequence whenever:
# (a) the observation is the first for a company, or
# (b) there is a gap of more than one year
df['start_seq'] = (df['year_diff'] > 1) | (df['year_diff'].isnull())

# Assign a sequence identifier within each company
df['group'] = df.groupby('company')['start_seq'].cumsum()

# Count the number of observations in each company-sequence
group_counts = df.groupby(['company', 'group']).size().reset_index(name='count')

# Merge sequence counts back into the dataset
df_merged = df.merge(group_counts,
                     on=['company', 'group'],
                     how='left')

# Remove sequences containing only a single observation,
# as they cannot be compared with a prior-year disclosure
result_df = df_merged[
    df_merged['count'].gt(1) | df_merged['count'].isna()
].drop(columns='count')

result_df.reset_index(drop=True, inplace=True)

# Recreate sequence identifiers after filtering
result_df['group'] = result_df.groupby('company')['start_seq'].cumsum()

# Create the previous-year disclosure column
manipulated_df = manipulate_data(result_df)

# Remove the first observation from each sequence because
# no prior-year disclosure exists for comparison
final_df = manipulated_df[~manipulated_df['start_seq'] == True]

# Drop temporary variables used during preprocessing
final_df.drop(['year_diff', 'start_seq'], axis=1, inplace=True)

# Reset the index of the final dataset
final_df.reset_index(drop=True, inplace=True)

In [ ]:
final_df

## This step transforms the dataset from the disclosure level to the sentence level. Each environmental disclosure is split into its constituent sentences using sentence-ending punctuation, and a separate observation is created for every sentence. Firm-year attributes and the corresponding prior-year disclosure are retained for each sentence, enabling subsequent classification of sentence-level disclosure characteristics such as novelty, specificity, and usefulness.

In [ ]:
# Convert disclosure-level observations into sentence-level observations.
# Each environmental disclosure is split into individual sentences, and
# a separate row is created for each sentence while retaining all
# associated firm-year information.

# Ensure the environmental disclosure field is stored as text
final_df['environment'] = final_df['environment'].astype(str)

c = 0
new_rows = []

# Iterate through each firm-year disclosure
for idx, row in final_df.iterrows():
    c += 1

    # Split the disclosure into individual sentences
    sentences = row['environment'].split('. ')

    # Create a new observation for each sentence
    for sentence in sentences:
        new_row = row.copy()
        new_row['sentence'] = sentence
        new_rows.append(new_row)

# Construct a sentence-level dataset
new_df = pd.DataFrame(new_rows)

# Reset the index after expanding the dataset
new_df.reset_index(drop=True, inplace=True)

# Retain variables required for subsequent analysis
new_df = new_df.loc[:, [
    'Accession.Number',
    'company',
    'year',
    'Market.Cap',
    'big.mktcap.median',
    'big.mktcap.mean',
    'sentence',
    'environment_previous'
]]

## Labeling the sentences

In [ ]:

# Ensure previous-year environmental disclosures are treated as strings
new_df['environment_previous'] = new_df['environment_previous'].astype(str)

x = 0

# Regex pattern used later to remove date-like expressions (e.g., "January 12", "Table 3")
date_pattern = r"\b(?:january|february|march|april|may|june|july|august|september|october|november|december|table)\b\s+\d+"

# Iterate over each sentence-level observation
for i in new_df.index:

    counter = 0  # counts capitalized words (used in specificity classification)

    # Current sentence
    s = new_df.loc[i, 'sentence']

    # -----------------------------
    # STEP 1: STANDARDIZE SENTENCE
    # -----------------------------
    env = new_df.loc[i, 'sentence']

    # Remove noisy numeric markers (e.g., "- 3 -")
    env = re.sub(r'- \d+ -', '', env)

    # Lowercase and normalize whitespace
    env = env.lower()
    env = remove_whitespace(env)

    # -----------------------------
    # STEP 2: PROCESS PRIOR-YEAR DISCLOSURE
    # -----------------------------
    # creating a list of all sentences in the previous year disclosure
    env_pre = new_df.loc[i, 'environment_previous'].split('. ')
    env_pre = [re.sub(r'- \d+ -', '', sentence) for sentence in env_pre]
    env_pre = [sentence.lower() for sentence in env_pre]
    env_pre = [remove_whitespace(sentence) for sentence in env_pre]

    print(i)

    # -----------------------------
    # STEP 3: REPEATED SENTENCE CLASSIFICATION
    # -----------------------------
    # Exact match with any prior-year sentence
    if env in env_pre:
        new_df.loc[i, 'repeated_0'] = 1
        new_df.loc[i, 'repeated_1word'] = 0
        new_df.loc[i, 'repeated_2words'] = 0
        new_df.loc[i, 'repeated_3words'] = 0
        new_df.loc[i, 'repeated_4words'] = 0

    else:
        # Token-level comparison if no exact match
        env_words = word_tokenize(env.lower())

        for a in env_pre:
            env_pre_words = word_tokenize(a.lower())

            # Allow approximate length similarity (±5 words)
            if len(env_words) == len(env_pre_words) or \
               len(env_words) - 5 < len(env_pre_words) < len(env_words) + 5 or \
               len(env_pre_words) - 5 < len(env_words) < len(env_pre_words) + 5:

                notsamewords = []

                # Identify words not shared between sentences
                for b in env_words:
                    if b not in env_pre_words:
                        notsamewords.append(b)

                # Classify repetition level based on number of differing words
                if len(notsamewords) < 1:
                    new_df.loc[i, 'repeated_0'] = 1
                    new_df.loc[i, 'repeated_1word'] = 0
                    new_df.loc[i, 'repeated_2words'] = 0
                    new_df.loc[i, 'repeated_3words'] = 0
                    new_df.loc[i, 'repeated_4words'] = 0
                    break

                elif len(notsamewords) == 1:
                    new_df.loc[i, 'repeated_1word'] = 1
                    new_df.loc[i, 'repeated_0'] = 0
                    new_df.loc[i, 'repeated_2words'] = 0
                    new_df.loc[i, 'repeated_3words'] = 0
                    new_df.loc[i, 'repeated_4words'] = 0
                    break

                elif len(notsamewords) == 2:
                    new_df.loc[i, 'repeated_2words'] = 1
                    new_df.loc[i, 'repeated_0'] = 0
                    new_df.loc[i, 'repeated_1word'] = 0
                    new_df.loc[i, 'repeated_3words'] = 0
                    new_df.loc[i, 'repeated_4words'] = 0
                    break

                elif len(notsamewords) == 3:
                    new_df.loc[i, 'repeated_3words'] = 1
                    new_df.loc[i, 'repeated_0'] = 0
                    new_df.loc[i, 'repeated_1word'] = 0
                    new_df.loc[i, 'repeated_2words'] = 0
                    new_df.loc[i, 'repeated_4words'] = 0
                    break

                elif len(notsamewords) == 4:
                    new_df.loc[i, 'repeated_4words'] = 1
                    new_df.loc[i, 'repeated_0'] = 0
                    new_df.loc[i, 'repeated_1word'] = 0
                    new_df.loc[i, 'repeated_2words'] = 0
                    new_df.loc[i, 'repeated_3words'] = 0
                    break

                else:
                    # No meaningful match
                    new_df.loc[i, 'repeated_0'] = 0
                    new_df.loc[i, 'repeated_1word'] = 0
                    new_df.loc[i, 'repeated_2words'] = 0
                    new_df.loc[i, 'repeated_3words'] = 0
                    new_df.loc[i, 'repeated_4words'] = 0

            else:
                # Sentence structure too different -> not considered repeated
                new_df.loc[i, 'repeated_0'] = 0
                new_df.loc[i, 'repeated_1word'] = 0
                new_df.loc[i, 'repeated_2words'] = 0
                new_df.loc[i, 'repeated_3words'] = 0
                new_df.loc[i, 'repeated_4words'] = 0

    # -----------------------------
    # STEP 4: SENTENCE CLEANING
    # -----------------------------

    # Guard against empty sentence strings
    if len(s) == 0:
        new_df.loc[i, 'sentence'] = s
        new_df.loc[i, 'specific'] = 0
        new_df.loc[i, 'count_specific_words'] = 0
        continue

    # Remove leading numbering if present
    if s[0].isdigit():
        s = s.split(' ', 1)[1] if ' ' in s else ''

    words = s.split()

    # Remove overly long tokens (likely noise or artifacts)
    filtered_words1 = [word for word in words if len(word) <= 15]
    cleaned_text = ' '.join(filtered_words1)

    # Standardize company name for later removal
    company_name = new_df.loc[i, 'company']
    company_words = [word.lower().capitalize() for word in company_name.split()]

    # Remove formatting artifacts and punctuation
    cleaned_text = re.sub(r'- \d+ -', '', cleaned_text)
    cleaned_text = cleaned_text.replace(',', '')
    cleaned_text = cleaned_text.replace("'", ' ')
    cleaned_text = cleaned_text.replace("(", ' ')
    cleaned_text = cleaned_text.replace(")", ' ')
    cleaned_text = cleaned_text.replace("-", ' ')
    cleaned_text = cleaned_text.replace("/", ' ')
    cleaned_text = cleaned_text.replace(";", ' ')
    cleaned_text = cleaned_text.replace("]", ' ')
    cleaned_text = cleaned_text.replace("[", ' ')
    # removed: cleaned_text.replace("", ' ')  -> this was inserting spaces between every character

    # Mask date-like patterns
    cleaned_text = re.sub(date_pattern, "xxxxdatexxxx", cleaned_text, flags=re.IGNORECASE)


    # Lower casing company name mentions (company names in disclosures does not give specific information, we lowercase them)
    for word in company_words:
        cleaned_text = cleaned_text.replace(word, word.lower())

    # Preserve country names with proper capitalization (if they are not capitalized, this step capitalizes country names)
    for country in countries:
        while country.lower() in cleaned_text:
            cleaned_text = cleaned_text.replace(country.lower(), country)

    # Guard against empty cleaned_text before indexing [0]
    if len(cleaned_text) == 0:
        new_df.loc[i, 'sentence'] = cleaned_text
        new_df.loc[i, 'specific'] = 0
        new_df.loc[i, 'count_specific_words'] = 0
        continue

    # Standardize sentence start
    cleaned_text = cleaned_text[0].lower() + cleaned_text[1:]

    # Remove short capitalized tokens (noise / abbreviations)
    cleaned_text = remove_123_words(cleaned_text)

    cleaned_text = cleaned_text + ' '

    # Force capitalization of selected domain-specific words
    for wordtocapitalize in wordstocapitalize:
        cleaned_text = cleaned_text.replace(wordtocapitalize, wordtocapitalize.capitalize())


    # Final whitespace cleanup
    cleaned_text = remove_whitespace(cleaned_text)

    # Remove leading digit if introduced during cleaning
    if len(cleaned_text) > 0 and cleaned_text[0].isdigit():
        cleaned_text = cleaned_text.split(' ', 1)[1] if ' ' in cleaned_text else ''

    # Store cleaned sentence
    new_df.loc[i, 'sentence'] = cleaned_text

    # -----------------------------
    # STEP 5: SPECIFICITY CLASSIFICATION
    # -----------------------------

    w = cleaned_text.split()

    # Detect sequential numbers (treated as non-specific structure. this sentences are more likely to be extracted from
    # tables which are not well-organized in the data)
    sequential_numbers = re.findall(r'\d+ \d+ \d+', cleaned_text)

    if len(sequential_numbers) > 0:
        new_df.loc[i, 'specific'] = 0
        numbers = ''

    else:
        # Count capitalized words as proxy for specificity
        for word in w:
            if word.istitle():
                counter += 1

        if counter > 0:
            x += 1
            new_df.loc[i, 'specific'] = 1
            numbers = re.findall(r'\d+', cleaned_text)

        else:
            # Fallback: detect meaningful numeric information
            numbers = re.findall(r'\d+', cleaned_text)
            matches = re.finditer(r'\d+', cleaned_text)

            digit = False
            for match in matches:
                if int(match.group()) > 9:
                    digit = True
                    break

            new_df.loc[i, 'specific'] = 1 if digit else 0

    # Count features used in specificity score
    new_df.loc[i, 'count_specific_words'] = counter + len(numbers)

    # Debug output for inspection
    if new_df.loc[i, 'specific'] == 1:
        print('###################################    ', numbers, counter)


## This step constructs the main dependent variable new.info, which identifies sentences that contain new informational content. A sentence is classified as new information if it is both specific and not repeated (neither exact repetitions nor near repetitions with only one-word differences from prior-year disclosures). The resulting binary variable is used to capture novel environmental disclosure at the sentence level.

In [ ]:
# Create a binary variable indicating whether a sentence contains
# new (non-repeated) and specific environmental information.
# A sentence is considered "new information" if:
# (1) it is classified as specific, and
# (2) it is not an exact or near-exact repetition of prior-year disclosures
#     (i.e., not in repeated_0 or repeated_1word categories).

new_df['new.info'] = (
    (new_df['specific'] == 1) &              # sentence contains specific information
    (new_df['repeated_0'] == 0) &            # not an exact repetition
    (new_df['repeated_1word'] == 0)          # not a near repetition (1-word difference)
)

# Convert boolean indicator into integer format (0/1) for analysis
new_df['new.info'] = new_df['new.info'].astype(int)

In [ ]:
new_df.to_csv('final_data.csv',index=0)